In [1]:
import sys
sys.path.append('..')

In [2]:
import torch
import torch.nn as N
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset,DataLoader
from src.utils.config import preprocessed_train_data,preprocessed_test_data
from src.utils.config import cat_cl
from models.model import ChurnMLP
import json
from sklearn.metrics import precision_score,recall_score,f1_score


Loading data from data mkdir

In [3]:
train_data=torch.load(preprocessed_train_data)
test_data=torch.load(preprocessed_test_data)


Get categorical(embeddings) , numerical features and targets from train & test

In [4]:
X_train_num, X_train_cat =train_data['X_num'],train_data['X_cat']
X_test_num, X_test_cat =test_data['X_num'],test_data['X_cat']

traintarg, testtarg = train_data['target'],test_data['target']

Initialize DataLoader for fututre Batches and dictionary with categories and their classes

In [5]:
train_dataset=TensorDataset(X_train_num,X_train_cat,traintarg)
X_train=DataLoader(train_dataset,batch_size=64,shuffle=True)

In [6]:
test_dataset=TensorDataset(X_test_num,X_test_cat,testtarg)
X_test=DataLoader(test_dataset,batch_size=256,shuffle=False)

In [7]:
with open(cat_cl,'r') as file:
    cat_classes=json.load(file)

Initialize model, pos_weight and optimizer (I'll choose Adam)

In [8]:
churn_predictor=ChurnMLP(num_lay_sz=X_train_num.shape[-1],cat_classes=cat_classes,emb_dim=8)

In [9]:
optimizer=optim.Adam(churn_predictor.parameters(),lr=5e-4)

In [10]:
pos_weight=len(train_data['target'][train_data['target']==0])/len(train_data['target'][train_data['target']==1])
pos_weight

2.768561872909699

Initalize BCEWithLogitsLoss

In [11]:
criterion=N.BCEWithLogitsLoss(pos_weight=None)

Train epochs 

In [15]:
for epoch in range(
    5):


    churn_predictor.train()
    total_train_loss= 0
    for x_num,x_cat,targ in X_train:


        optimizer.zero_grad()

        logits=churn_predictor(x_num,x_cat)
        batch_loss=criterion(logits,targ)
        batch_loss.backward()
        optimizer.step()

        total_train_loss += batch_loss.item() * targ.size(0) 

    avg_train_loss=total_train_loss/len(train_dataset)


    churn_predictor.eval()
    correct=0
    total=0

    f1,prec,rec=0,0,0

    with torch.no_grad():
        for x_num,x_cat,targ in X_test:
            logits=churn_predictor(x_num,x_cat)
            probs=torch.sigmoid(logits)
            preds=(probs>0.35).float()


            correct+= (preds==targ).sum().item()
            total+=targ.size(0)

            f1=f1_score(y_pred=preds,y_true=targ)
            prec=precision_score(y_pred=preds,y_true=targ)
            rec=recall_score(y_pred=preds,y_true=targ)

    print(f"Epoch: {epoch+1}| Train_Avg_loss: {avg_train_loss} | Test_Accuracy: {correct/total} | Recall: {rec} | Precision: {prec} | F1: {f1}")

Epoch: 1| Train_Avg_loss: 0.3770241295779306 | Test_Accuracy: 0.7629524485450674 | Recall: 0.6153846153846154 | Precision: 0.5333333333333333 | F1: 0.5714285714285714
Epoch: 2| Train_Avg_loss: 0.3718230087028398 | Test_Accuracy: 0.7672107877927609 | Recall: 0.5897435897435898 | Precision: 0.5348837209302325 | F1: 0.5609756097560976
Epoch: 3| Train_Avg_loss: 0.37364530301513 | Test_Accuracy: 0.7508871540099361 | Recall: 0.6666666666666666 | Precision: 0.5416666666666666 | F1: 0.5977011494252874
Epoch: 4| Train_Avg_loss: 0.37314848574840476 | Test_Accuracy: 0.759403832505323 | Recall: 0.6410256410256411 | Precision: 0.5319148936170213 | F1: 0.5813953488372093
Epoch: 5| Train_Avg_loss: 0.3705424602187437 | Test_Accuracy: 0.765791341376863 | Recall: 0.6153846153846154 | Precision: 0.5454545454545454 | F1: 0.5783132530120482


Saving model


In [16]:
torch.save(churn_predictor.state_dict(), "../loaded_models/churn_model.pth")
print("Веса сохранены!")

Веса сохранены!
